# Cross-Layer Alignment

How aligned are representations between layers? Adjacent cosine,
full alignment matrix, component alignment, and early-late preservation.

## Why This Matters

Layer cross alignment characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.cross_layer_alignment import (
    adjacent_layer_alignment, full_layer_alignment_matrix,
    component_alignment_across_layers, early_late_alignment,
    cross_layer_alignment_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Adjacent Layer Alignment

Cosine similarity between consecutive layers' residual streams.
High = layer barely changes direction; low = major redirect.

In [ ]:
result = adjacent_layer_alignment(model, tokens, position=-1)
print(f"Mean alignment: {result['mean_alignment']:.4f}")
for p in result['per_pair']:
    flag = ' [ALIGNED]' if p['is_aligned'] else ''
    bar = '█' * int((p['cosine'] + 1) * 20)
    print(f"  Layers {p['layers']}: cos={p['cosine']:.4f} {bar}{flag}")

## Full Alignment Matrix

Cosine similarity between all pairs of layers reveals block
structure and long-range alignment patterns.

In [ ]:
result = full_layer_alignment_matrix(model, tokens, position=-1)
print(f"Alignment matrix ({result['n_layers']}x{result['n_layers']}):")
for i, row in enumerate(result['alignment_matrix']):
    vals = ' '.join(f'{v:+.3f}' for v in row)
    print(f"  L{i}: [{vals}]")

## Component Alignment Across Layers

Do attention and MLP outputs point in similar directions across layers?

In [ ]:
result = component_alignment_across_layers(model, tokens, position=-1)
print("Attention alignment:")
for p in result['attn_alignment']:
    print(f"  Layers {p['layers']}: cos={p['cosine']:.4f}")
print("MLP alignment:")
for p in result['mlp_alignment']:
    print(f"  Layers {p['layers']}: cos={p['cosine']:.4f}")

## Early-Late Alignment

How much does the early representation influence the final output?

In [ ]:
result = early_late_alignment(model, tokens, position=-1)
print(f"Early-late cosine: {result['early_late_cosine']:.4f}")
print(f"Early norm: {result['early_norm']:.4f}")
print(f"Late norm: {result['late_norm']:.4f}")
print(f"Norm growth: {result['norm_growth']:.4f}x")
preserved = 'YES' if result['is_preserved'] else 'NO'
print(f"Direction preserved: {preserved}")

## Alignment Summary

In [ ]:
result = cross_layer_alignment_summary(model, tokens, position=-1)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")